In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [11]:
import numpy as np
import pandas as pd
import mlflow

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV

from scipy.stats import randint, uniform
from xgboost import XGBRegressor

#warnings.filterwarnings('ignore')

In [3]:
!pip install kaggle

In [7]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/cs231n/assignments/4/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [8]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:00<00:00, 191MB/s]



In [9]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [12]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
stores = pd.read_csv('stores.csv')
features = pd.read_csv('features.csv')

In [15]:
print(train_full.shape)
print(train_full.columns)

(421570, 16)
Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Type', 'Size',
       'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3',
       'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment'],
      dtype='object')


In [14]:
train_full = train.merge(stores, on="Store", how="left")
train_full = train_full.merge(features, on=["Store", "Date", "IsHoliday"], how="left")

In [16]:
from sklearn.model_selection import train_test_split

X = train_full.drop('Weekly_Sales', axis=1)
y = train_full['Weekly_Sales']

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2,
    random_state=42,
)

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin

class MissingValueImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy="median"):
        self.strategy = strategy

    def fit(self, X, y=None):
        if self.strategy == "median":
            self.fill_values_ = X.median(numeric_only=True)
        elif self.strategy == "mean":
            self.fill_values_ = X.mean(numeric_only=True)
        elif self.strategy == "most_frequent":
            self.fill_values_ = X.mode().iloc[0]
        else:
            raise ValueError("Unsupported strategy")

        return self

    def transform(self, X):
        X = X.copy()
        return X.fillna(self.fill_values_)

In [19]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=fc6e84cf-e335-4dcd-b68c-1886f8c49cfc&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=ddc1fc3d8dc20bd20a90467bb8f8eef2bbd4c64d621f1cdec9d87e17fe48c425




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [33]:
import torch
print(torch.cuda.is_available())

True


In [63]:
from sklearn.base import BaseEstimator, TransformerMixin

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        self.to_drop_ = [
            col for col in upper.columns
            if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.drop(columns=self.to_drop_, errors="ignore")

In [25]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['Store', 'Dept', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment']
Categorical: ['Date', 'IsHoliday', 'Type']


In [26]:
encoder_config = {
    "handle_unknown": "ignore",
    "drop": None,
    "min_frequency": 5,
    "max_categories": 100,
    "sparse_output": False
}

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown=encoder_config["handle_unknown"],
        drop=encoder_config["drop"],
        min_frequency=encoder_config["min_frequency"],
        max_categories=encoder_config["max_categories"],
        sparse_output=encoder_config["sparse_output"]
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [27]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    random_state=42
)
param_distributions = {
    "model__n_estimators": randint(500, 2000),
    "model__max_depth": randint(3, 10),
    "model__learning_rate": uniform(0.01, 0.15),

    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),

    "model__min_child_weight": randint(1, 10),
    "model__gamma": uniform(0, 3),

    "model__reg_alpha": uniform(0, 1),
    "model__reg_lambda": uniform(1, 4)
}
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb)
])
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

In [44]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGBoost_final_model_run"):

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_

    preds = best_model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)

    mlflow.sklearn.log_model(
        best_model,
        name="model",
        serialization_format="pickle"
    )

    print("DONE SAVING MODEL")

Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV] END model__colsample_bytree=0.749816047538945, model__gamma=2.8521429192297485, model__learning_rate=0.11979909127171076, model__max_depth=7, model__min_child_weight=5, model__n_estimators=621, model__reg_alpha=0.15599452033620265, model__reg_lambda=1.2323344486727978, model__subsample=0.9464704583099741; total time=   7.5s
[CV] END model__colsample_bytree=0.749816047538945, model__gamma=2.8521429192297485, model__learning_rate=0.11979909127171076, model__max_depth=7, model__min_child_weight=5, model__n_estimators=621, model__reg_alpha=0.15599452033620265, model__reg_lambda=1.2323344486727978, model__subsample=0.9464704583099741; total time=   5.7s
[CV] END model__colsample_bytree=0.749816047538945, model__gamma=2.8521429192297485, model__learning_rate=0.11979909127171076, model__max_depth=7, model__min_child_weight=5, model__n_estimators=621, model__reg_alpha=0.15599452033620265, model__reg_lambda=1.2323344486727978, mo

2026/07/03 12:00:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


DONE SAVING MODEL
🏃 View run XGBoost_final_model_run at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/4fdefd6ac3b743dba009a8bbcb8489ea
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2


In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def build_preprocessor(numeric_features, categorical_features):

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            min_frequency=10,
            sparse_output=True
        ))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    return preprocessor

In [41]:
encoder_runs = [
    {
        "handle_unknown": "ignore",
        "drop": None,
        "min_frequency": None,
        "max_categories": None,
        "sparse_output": True
    },
    {
        "handle_unknown": "ignore",
        "drop": "first",
        "min_frequency": None,
        "max_categories": None,
        "sparse_output": True
    },
    {
        "handle_unknown": "ignore",
        "drop": "first",
        "min_frequency": 10,
        "max_categories": None,
        "sparse_output": True
    },
    {
        "handle_unknown": "ignore",
        "drop": "first",
        "min_frequency": 10,
        "max_categories": 50,
        "sparse_output": True
    },
    {
        "handle_unknown": "ignore",
        "drop": None,
        "min_frequency": 5,
        "max_categories": 100,
        "sparse_output": False
    }
]

In [46]:
mlflow.set_experiment("XGBoost_Training")

for i, cfg in enumerate(encoder_runs):

    with mlflow.start_run(run_name=f"XGBoost_Run{i}_OneHot"):

        # Build preprocessing
        preprocessor = build_onehot_preprocessor(
            numeric_features,
            categorical_features,
            handle_unknown=cfg["handle_unknown"],
            drop=cfg["drop"],
            min_frequency=cfg["min_frequency"],
            max_categories=cfg["max_categories"],
            sparse_output=cfg["sparse_output"]
        )

        # Model pipeline
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", XGBRegressor(
                random_state=42,
                objective="reg:squarederror",
                tree_method="hist",
                device="cuda"
            ))
        ])

        # Train
        pipeline.fit(X_train, y_train)

        # Predict
        preds = pipeline.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)

        # Log params
        mlflow.log_params(cfg)

        # Log metrics
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)

        print(f"Run {i}: RMSE={rmse:.4f}, MAE={mae:.4f}")

Run 0: RMSE=4479.0970, MAE=2167.5545
🏃 View run XGBoost_Run0_OneHot at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/9e0a7ee1b1aa4bc7b17e138874ea89f6
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2
Run 1: RMSE=4479.0970, MAE=2167.5545
🏃 View run XGBoost_Run1_OneHot at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/cccbb1d1a5cc4242a5ea66e68f1d0635
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2
Run 2: RMSE=4479.0970, MAE=2167.5545
🏃 View run XGBoost_Run2_OneHot at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/c4afaae8ada34322ae6456bddaec40f8
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2
Run 3: RMSE=4479.0970, MAE=2

In [49]:
mlflow.set_experiment("XGBoost_Training")

for i, cfg in enumerate(encoder_runs):
   with mlflow.start_run(run_name=f"XGBoost_Run{i}_OneHot_debuged"):
    preprocessor = build_onehot_preprocessor(
        numeric_features,
        categorical_features,
        handle_unknown=cfg["handle_unknown"],
        drop=cfg["drop"],
        min_frequency=cfg["min_frequency"],
        max_categories=cfg["max_categories"],
        sparse_output=cfg["sparse_output"]
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            random_state=42,
            objective="reg:squarederror",
            tree_method="hist",
            device="cuda"
        ))
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)
    # Log params
    mlflow.log_params(cfg)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    print(f"Run {i}: RMSE={rmse:.4f}, MAE={mae:.4f}")

Run 0: RMSE=6854.4271, MAE=3586.3739
🏃 View run XGBoost_Run0_OneHot_debuged at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/a1ea104687564eefb2ca43c0425f00cc
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2
Run 1: RMSE=6863.4015, MAE=3547.7079
🏃 View run XGBoost_Run1_OneHot_debuged at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/bd692ec5de1a4a98bc84c86d76ba365a
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2
Run 2: RMSE=6863.4015, MAE=3547.7079
🏃 View run XGBoost_Run2_OneHot_debuged at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/c8d1ffaef4354a55a66d3c0c998cc5e2
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2
Run 

In [37]:
mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGBoost_run1_median"):

    preprocessor = get_preprocessor(num_strategy="median")
    random_search = get_random_search(preprocessor)

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)

    mlflow.log_param("imputer", "median")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
🏃 View run XGBoost_run1_median at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/f2b075641f41474dba9716768e07b361
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2


In [38]:
with mlflow.start_run(run_name="XGBoost_run2_mean"):

    preprocessor = get_preprocessor(num_strategy="mean")
    random_search = get_random_search(preprocessor)

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)

    mlflow.log_param("imputer", "mean")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
🏃 View run XGBoost_run2_mean at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/b6abb68692f2407587ad4cd3313bea2d
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2


In [39]:
with mlflow.start_run(run_name="XGBoost_run3_most_frequent"):

    preprocessor = get_preprocessor(num_strategy="most_frequent")
    random_search = get_random_search(preprocessor)

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)

    mlflow.log_param("imputer", "most_frequent")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
🏃 View run XGBoost_run3_most_frequent at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/e9e1452708794989be2686f78f4fad08
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2
